In [ ]:
import struct
from pathlib import Path

import numpy as np

# 修改为需要查看的轨迹文件路径
bin_path = Path("plummer_256.bin")

with bin_path.open("rb") as file:
    header = file.read(8)
    if len(header) != 8:
        raise ValueError("BIN 文件头不足 8 字节，文件可能损坏")

    particle_count, record_count = struct.unpack("<ii", header)
    raw_data = np.fromfile(file, dtype="<f4")

expected_values = record_count * particle_count * 3
if particle_count <= 0 or record_count <= 0:
    raise ValueError(f"非法文件头：P={particle_count}, R={record_count}")
if raw_data.size != expected_values:
    raise ValueError(
        f"数据长度不匹配：预期 {expected_values} 个 float32，"
        f"实际 {raw_data.size} 个"
    )

# 文件按粒子顺序存储：[粒子 P, 记录 R, 坐标(x, y, z)]
particle_major = raw_data.reshape(particle_count, record_count, 3)
# 分析和绘图通常使用：[记录 R, 粒子 P, 坐标(x, y, z)]
trajectory = particle_major.transpose(1, 0, 2)

print(f"文件：{bin_path.resolve()}")
print(f"文件大小：{bin_path.stat().st_size:,} 字节")
print(f"粒子数 P：{particle_count}")
print(f"记录数 R：{record_count}")
print(f"文件布局：[P, R, xyz] = {particle_major.shape}")
print(f"分析布局：[R, P, xyz] = {trajectory.shape}")
print(f"是否包含 NaN：{np.isnan(trajectory).any()}")
print(f"是否包含 Inf：{np.isinf(trajectory).any()}")
print("坐标最小值 [x, y, z]：", trajectory.min(axis=(0, 1)))
print("坐标最大值 [x, y, z]：", trajectory.max(axis=(0, 1)))

frames_to_show = min(3, record_count)
particles_to_show = min(5, particle_count)
for frame in range(frames_to_show):
    print(f"\n第 {frame} 帧，前 {particles_to_show} 个粒子的 [x, y, z]：")
    print(trajectory[frame, :particles_to_show])